[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/warmup/NB_Warmup_03_ML_I_Recap.ipynb)

# NB Warm-up 03 — Models, estimators, and predictions

**STAT3009 · Recommender Systems**  
**Suggested time:** 15–18 minutes

> **Recall → Run → Explain.** Review the core ideas from Machine Learning I before today's lesson on validation and model selection.

**Classroom route:** quick recall (2 min) → `LinearRegression` (3 min) → global-mean RS (3 min) → std-aware three-mean weights (6–8 min) → bridge question (1 min). All data are included below; no download is needed.


## 1. Ninety-second recall

Decide whether each statement is true or false **before** opening the answers.

1. In `LinearRegression(fit_intercept=True)`, `fit_intercept` is chosen before `fit`.
2. `coef_` and `intercept_` are learned from the training examples.
3. `predict(X_new)` needs both `X_new` and its unknown answers.
4. A small **training** RMSE proves that the model will predict new rows well.

<details>
<summary><b>Reveal the answers</b></summary>

1. **True.** `fit_intercept` is a constructor setting.
2. **True.** These fitted attributes appear after `fit(X, y)`.
3. **False.** `predict` receives `X_new` only.
4. **False.** Training RMSE measures fit on rows the model learned from. We need held-out rows to check generalization.

</details>


## 2. The four questions in one familiar model

For linear regression, complete this table aloud. The answers are just below it.

| Question | Your recall |
|---|---|
| Model | What rule turns features into a prediction? |
| Hyperparameter | Which setting did we choose before fitting? |
| Fitting criterion | What error does `fit` minimize on training rows? |
| Learned parameters | Which values appear after fitting? |

<details>
<summary><b>Reveal the answers</b></summary>

- **Model:** $\widehat y=b+x^{\mathsf T}w$.
- **Hyperparameter in this example:** `fit_intercept=True`.
- **Fitting criterion:** sum of squared training errors.
- **Learned parameters:** $\widehat b$ and $\widehat w$, stored in `intercept_` and `coef_`.

</details>


## 3. Run the estimator workflow

The data follow the same contract as ML I: each row of `X_train` has one matching target in `y_train`. `X_new` contains feature rows whose answers we want to predict.


In [1]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.utils.validation import check_is_fitted

X_train = np.array([[0.0], [1.0], [2.0], [3.0]])
y_train = np.array([1.0, 3.0, 5.0, 7.0])
X_new = np.array([[4.0], [5.0]])

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_new shape:', X_new.shape)


X_train shape: (4, 1)
y_train shape: (4,)
X_new shape: (2, 1)


### Your turn — three familiar calls

Create a `LinearRegression` object with an intercept, fit it, and predict the two new rows. Use the three-line pattern from ML I; then compare with the solution.


In [11]:
# linear_model = LinearRegression(fit_intercept=True)
# linear_model.fit(..., ...)
# new_predictions = linear_model.predict(...)
# print(new_predictions)

linear_model = LinearRegression(fit_intercept=True)
linear_model.fit(X_train, y_train)
new_predictions = linear_model.predict(X_new)


### Solution


In [14]:
linear_model = LinearRegression(fit_intercept=True)
linear_model.fit(X_train, y_train)
new_predictions = linear_model.predict(X_new)

print('predictions for x=4 and x=5:', new_predictions)
print('fit_intercept:', linear_model.get_params()['fit_intercept'])
print('coef_:', linear_model.coef_)
print('intercept_:', linear_model.intercept_)


predictions for x=4 and x=5: [ 9. 11.]
fit_intercept: True
coef_: [2.]
intercept_: 1.0


`fit_intercept` was available before training. `coef_` and `intercept_` appeared after training. The learned rule is approximately $\widehat y=1+2x$, so the new predictions should be 9 and 11.


## 4. The same interface for a recommender baseline

This is the `GlobalMeanRS` pattern from ML I. Read the two methods; you do **not** need to rewrite the class. Its model predicts one constant rating, and `fit` learns that constant from `y`.


In [15]:
class GlobalMeanRS(BaseEstimator, RegressorMixin):
    def fit(self, X, y):
        self.global_mean_ = np.mean(y)
        return self
    
    def predict(self, X):
        check_is_fitted(self, 'global_mean_')
        return np.full(shape=(X.shape[0],), fill_value=self.global_mean_, dtype=float)

In [18]:
X_train = np.array([[1, 2], [1, 3], [2, 1], [2, 3]])
y_train = np.array([4.5, 4.7, 4.6, 4.8])
X_test = np.array([[1, 2], [2, 3]])

rs_model = GlobalMeanRS()
rs_model.fit(X_train, y_train)
y_pred = rs_model.predict(X_test)

In [19]:
y_pred

array([4.65, 4.65])

### Your turn — reuse the workflow

Fit `GlobalMeanRS` on `X_ratings, y_ratings`, then predict both rows of `X_pairs_new`. Before running the code, what value should both predictions have?


In [ ]:
# rs_model = GlobalMeanRS()
# rs_model.fit(..., ...)
# rs_predictions = rs_model.predict(...)
# print(rs_predictions)


### Solution


In [ ]:
rs_model = GlobalMeanRS()
rs_model.fit(X_ratings, y_ratings)
rs_predictions = rs_model.predict(X_pairs_new)

print('learned global_mean_:', rs_model.global_mean_)
print('new-pair predictions:', rs_predictions)
print('training RMSE:', root_mean_squared_error(
    y_ratings, rs_model.predict(X_ratings)
))


## 5. Let user spread change the three-mean weights

Last lesson, we predicted with the global, user, and item means **separately**. Now combine those three familiar predictions. Give the global mean a fixed weight; let each user's rating standard deviation decide how to split the remaining weight between the user and item means. A user whose ratings vary more gets less weight on their own mean.

Apply the four questions before writing code:

**1 · Model.** For a user–item pair, combine the three means:

$$\widehat r_{ui}=w_u(u)\bar r_u+w_i(u)\bar r_i+w_0\mu.$$

Set $w_0=0.2$. From the user's rating standard deviation $s_u$, calculate

$$q_u=\frac{1}{1+s_u^2},\qquad
w_u(u)=(1-w_0)q_u,\qquad
w_i(u)=(1-w_0)(1-q_u).$$

The three weights are nonnegative and add to **1**. A new user has no $s_u$, so use $q_u=0$; an unseen item's mean falls back to $\mu$.

**2 · Hyperparameter.** `global_weight` ($w_0$) is chosen before `fit`. The means and $s_u$ are learned from training ratings; they are **not** hyperparameters.

**3 · Loss / fitting criterion.** `fit` calculates means and standard deviations from training rows. The rule for $q_u$ is a **candidate heuristic**, not a solution to a training-loss minimization problem. We will use held-out RMSE in ML II to check whether it helps.

**4 · Learned parameters.** `global_mean_`, `user_means_`, `user_stds_`, and `item_means_`.

With very few ratings, a user's sample std can be misleadingly small. This example shows the interface and the idea; it does not claim these weights are universally best.


### Same user mean, different weights

Users 0 and 1 below both have mean rating **3**, but their ratings are $(2,4)$ and $(1,5)$, giving population standard deviations **1** and **2**. Item 2 was rated by two other users and has mean **4.5**; the global mean is **3.9**.

With $w_0=0.2$, calculate $q_u$, then the three weights for users 0 and 1. Which user will lean more toward item 2's mean?


In [ ]:
X_std = np.array([
    [0, 0], [0, 1],
    [1, 0], [1, 1],
    [2, 0], [2, 1], [2, 2],
    [3, 0], [3, 1], [3, 2],
])
y_std = np.array([2.0, 4.0, 1.0, 5.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0])
X_std_new = np.array([[0, 2], [1, 2], [4, 2]])


### Turn the rule into a sklearn-style estimator

`fit(X, y)` stores the three means and each user's standard deviation. `predict(X_new)` looks up those values, computes weights **for that user**, then returns one rating prediction. The item column stays an input as in the previous RS estimators.


In [ ]:
class StdWeightedThreeMeanRS(RegressorMixin, BaseEstimator):
    def __init__(self):

    def fit(self, X, y):

        return self

    def predict(self, X):
        check_is_fitted(self, ['global_mean_', 'user_means_',
                               'user_stds_', 'item_means_'])

        return np.array(predictions)


### Your turn — two users, one item

For user 0, $s_0=1$ gives weights $(w_u,w_i,w_0)=(0.4,0.4,0.2)$. For user 1, $s_1=2$ gives $(0.16,0.64,0.2)$. Predict `(0, 2)` and `(1, 2)` by hand before running the cell. What should happen for the unseen user in `(4, 2)`?


The predictions are **3.78**, **4.14**, and **4.38**. Users 0 and 1 have the *same* user mean, yet user 1's larger std gives more weight to item 2's mean. The unseen user has $q_u=0$ and uses only the item and global means. Whether this std-based rule actually improves predictions is a question for held-out validation.


## Exit ticket — bridge to Machine Learning II

`LinearRegression`, `GlobalMeanRS`, and `StdWeightedThreeMeanRS` have different prediction rules but share `fit(X, y)` and `predict(X_new)`.

**Question:** The std-aware weights are plausible, but how could we check whether they predict new user–movie pairs better than fixed weights **without using the final test answers**?

<details>
<summary><b>After discussion</b></summary>

Compare both weighting rules on ratings that were **held out from fitting**. Today's lesson develops holdout validation and cross-validation for that choice.

</details>
